In [1]:
import os
from PIL import Image

# Define the directory
directory = r"C:\Users\Tassili\Desktop\GP\autoEncoder\driv\newTrain"

# Iterate over the files in the directory
for filename in os.listdir(directory):
    # Full path of the file
    file_path = os.path.join(directory, filename)
    
    # Check if the file is an image
    if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
        try:
            # Open the image
            with Image.open(file_path) as img:
                width, height = img.size
                # Check if the width or height is less than 256
                if width < 256 or height < 256:
                    # Close the image and delete the file
                    img.close()
                    os.remove(file_path)
                    print(f"Deleted: {file_path}")
        except Exception as e:
            print(f"Error processing file {file_path}: {e}")


In [2]:
import torch
from torch import nn
import requests
import zipfile
from pathlib import Path
import os
import cv2
import glob
import numpy as np
import argparse
import time
import math
import random
import shutil
import sys
import glob

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader
from torchvision import transforms


from pathlib import Path

from PIL import Image
from torch.utils.data import Dataset

device = "cuda" if torch.cuda.is_available() else "cpu"
device


'cpu'

In [3]:
# ################################################
# ImageFolder Dataset

class ImageFolder(Dataset):
    """Load an image folder database. Training and testing image samples
    are respectively stored in separate directories:

    .. code-block::

        - rootdir/
            - train/
                - img000.png
                - img001.png
                ...
            - valid/
                - img000.png
                - img001.png
                ...

    Args:
        root (string): root directory of the dataset
        transform (callable, optional): a function or transform that takes in a
            PIL image and returns a transformed version
    """

    def __init__(self, root, transform=None):
        splitdir = Path(root)

        if not splitdir.is_dir():
            raise RuntimeError(f'Invalid directory "{root}"')

        self.samples = [f for f in splitdir.iterdir() if f.is_file()]

        self.transform = transform

    def __getitem__(self, index):
        """
        Args:
            index (int): Index

        Returns:
            img: `PIL.Image.Image` or transformed `PIL.Image.Image`.
        """
        img = Image.open(self.samples[index]).convert("RGB")
        if self.transform:
            return self.transform(img)
        return img

    def __len__(self):
        return len(self.samples)

In [4]:
# ################################################
# SequenceFolder Dataset

class SequenceFolder(Dataset):
    """Load an image folder database. Training and testing image samples
    are respectively stored in separate directories:

    .. code-block::

        - rootdir/
            - train/
              - video 1
                - img000.png
                - img001.png
                ...
              - video 2
                - img000.png
                - img001.png
                ...
            - test/
              - video 1
                - img000.png
                - img001.png
                ...
              - video 2
                - img000.png
                - img001.png
                ...

    Args:
        root (string): root directory of the dataset
        transform (callable, optional): a function or transform that takes in a
            PIL image and returns a transformed version
        split (string): split mode ('train' or 'val')
    """

    def __init__(self, root, transform=None, split="train"):
        self.mode = split
        splitdir = Path(root) / split

        if not splitdir.is_dir():
            raise RuntimeError(f'Invalid directory "{root}"')

        self.samples = self.get_all_images(splitdir)

        self.transform = transform

    def get_all_images(self, direc):
        self.images = []
        self.image_sequence = []
        self.sequences = [f for f in direc.iterdir() if f.is_dir()]
        for sd in self.sequences:
            images = []
            for f in (sd / 'img').iterdir():
                if f.is_file():
                  images.append(f)
            self.image_sequence.append(images)
            self.images = self.images + list(sorted(images))

        return self.images

    def __getitem__(self, index):
        """
        Args:
            index (int): Index

        Returns:
            img: `PIL.Image.Image` or transformed `PIL.Image.Image`.
        """
        img = Image.open(self.samples[index]).convert("RGB")
        if self.transform:
            return self.transform(img)
        return img

    def __len__(self):
        return len(self.samples)



In [5]:
seed = 123                                        # for reproducibility
cuda = True                                       # use GPU
save = True                                       # save trained model
image_dataset = r'C:\Users\Tassili\Desktop\GP\autoEncoder\driv\newTrain'  # path to the root of the image dataset

sequence_dataset = r'C:\Users\Tassili\Desktop\GP\autoEncoder\driv\Video'  # path to the root of the video dataset
checkpoint = r'checkpoint_90percent_new_daset.pth.tar'                                   # load pretrained model
epochs = 100                                    # total training epochs
clip_max_norm = 1.0                               # avoid gradient explosion
patch_size = (256,256)                          # input size for the training network
learning_rate = 1e-3
batch_size = 16
test_batch_size = 16
num_workers =0                         # multi-process for loading training data
N = 45
M = 60

In [6]:

class Loss(nn.Module):

    def __init__(self):
        super().__init__()
        self.mse = nn.MSELoss()

    def forward(self, output, target):
        out = {}
        out["mse_loss"] = self.mse(output["x_hat"], target)
        out["loss"] = out["mse_loss"] * 255

        return out

In [7]:

class AverageMeter:
    """Compute running average."""

    def __init__(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count

In [8]:
def configure_optimizers(net, learning_rate):

    optimizer = optim.Adam(
        net.parameters(),
        lr=learning_rate,
    )

    return optimizer

In [9]:
def save_checkpoint(state, is_best, filename="checkpoint.pth.tar"):
    torch.save(state, filename)
    shutil.copyfile(filename, "checkpoint90percent_new_dataset.pth.tar")

In [10]:
def PSNR(img1, img2):
    # img1 and img2 within range [0, 1]
    # img1 shape: (B, C, H, W)
    # img2 shape: (B, C, H, W)

    img1, img2 = img1.detach(), img2.detach()
    img1 = img1 * 255
    img2 = img2 * 255
    batch_size = img1.shape[0]
    img1 = img1.reshape(batch_size, -1)
    img2 = img2.reshape(batch_size, -1)
    mse = torch.mean((img1 - img2) ** 2)
    return torch.mean(20 * torch.log10(255.0 / torch.sqrt(mse)))

In [11]:

def train_one_epoch(
    model, criterion, train_dataloader, optimizer, epoch, clip_max_norm
):
    model.train()
    device = next(model.parameters()).device

    for i, d in enumerate(train_dataloader):
        d = d.to(device)

        optimizer.zero_grad()

        out_net = model(d)

        out_criterion = criterion(out_net, d)
        out_criterion["loss"].backward()
        if clip_max_norm > 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), clip_max_norm)
        optimizer.step()

        if i % 10 == 0:
            print(
                f"Train epoch {epoch}: ["
                f"{i*len(d)}/{len(train_dataloader.dataset)}"
                f" ({100. * i / len(train_dataloader):.0f}%)]"
                f'\tLoss: {out_criterion["loss"].item():.3f} |'
                f'\tMSE loss: {out_criterion["mse_loss"].item():.3f}'
            )


In [12]:


def test_epoch(epoch, test_dataset, transform, model, criterion):
    model.eval()
    device = next(model.parameters()).device

    mse_loss = AverageMeter()
    psnr = AverageMeter()

    with torch.no_grad():
        for video in test_dataset.image_sequence:
            psnr_video = AverageMeter()
            for image in video:
                image = Image.open(image).convert("RGB")
                image = transform(image)
                image = image.to(device).unsqueeze(0)

                out_net = model(image)
                d_out = out_net['x_hat']
                out_criterion = criterion(out_net, image)
                psnr_video.update(PSNR(image, d_out))

            mse_loss.update(out_criterion["mse_loss"])
            psnr.update(psnr_video.avg)

    print(
        f"Test epoch {epoch}: Average losses:"
        f"\tMSE loss: {mse_loss.avg:.3f}"
        f'\tSequence-wise PSNR: {psnr.avg: .3f}\n'
    )

    return mse_loss.avg,psnr.avg

In [13]:
# network structure define
def conv(in_channels, out_channels, kernel_size=6, stride=2):
    return nn.Conv2d(
        in_channels,
        out_channels,
        kernel_size=kernel_size,
        stride=stride,
        padding=kernel_size // 2,
    )


def deconv(in_channels, out_channels, kernel_size=6, stride=2):
    return nn.ConvTranspose2d(
        in_channels,
        out_channels,
        kernel_size=kernel_size,
        stride=stride,
        output_padding=stride - 1,
        padding=kernel_size // 2,
    )


class Network(nn.Module):

    def __init__(self,N, M, init_weights=True, **kwargs):
        super().__init__(**kwargs)

        self.g_a = nn.Sequential(
            conv(3, N),
            # nn.PReLU(),
            # nn.Conv2d(40, 40, kernel_size=1),
            nn.Conv2d(N, N, kernel_size=1),
            conv(N, N),
            nn.Conv2d(N, N, kernel_size=1),
            conv(N, N),
            nn.Conv2d(N, N, kernel_size=1),
            # nn.PReLU(),
            conv(N, 44),
        )


        self.g_s = nn.Sequential(
            deconv(44, N),
            deconv(N, N),
            deconv(N, N),
            nn.ConvTranspose2d(N, 3, kernel_size=3, stride=2, padding =2 , output_padding=1),

        )

        self.N = N
        self.M = M

        if init_weights:
            self._initialize_weights()

    def forward(self, x):
        y = self.g_a(x)
        x_hat = self.g_s(y)
        return {
            "x_hat": x_hat,
            # "x_quan": quan,
        }


    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
                nn.init.kaiming_normal_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def compress(self, x):
        y = self.g_a(x)
        return y

    def decompress(self, y_hat):
        x_hat = self.g_s(y_hat).clamp_(0, 1) # Limiting. Limit the value of input to [min, max], output as a tensor
        return {"x_hat": x_hat}



In [14]:
import torch
import torch.nn as nn
i= torch.randn(1, 3, 256, 256)
x1=conv(3, N,kernel_size=6)
x2=nn.Conv2d(N, N, kernel_size=1)
x3=conv(N, N,kernel_size=6)
x4=nn.Conv2d(N, N, kernel_size=1)
x5=conv(N, N,kernel_size=6)
x6=nn.Conv2d(N, N, kernel_size=1)
x7=conv(N,44,kernel_size=6)
i=x1(i)
print(i.size())
i=x2(i)
print(i.size())
i=x3(i)
print(i.size())
i=x4(i)
print(i.size())
i=x5(i)
print(i.size())
i=x6(i)
print(i.size())
i=x7(i)
print(i.size())
x1=deconv(44, N,kernel_size=6)
x2=deconv(N, N,kernel_size=6)
x3=deconv(N, N,kernel_size=6)
x4=nn.ConvTranspose2d(N, 3, kernel_size=3, stride=2, padding =2 , output_padding=1)

i=x1(i)
print(i.size())
i=x2(i)
print(i.size())
i=x3(i)
print(i.size())
i=x4(i)
print(i.size())


torch.Size([1, 45, 129, 129])
torch.Size([1, 45, 129, 129])
torch.Size([1, 45, 65, 65])
torch.Size([1, 45, 65, 65])
torch.Size([1, 45, 33, 33])
torch.Size([1, 45, 33, 33])
torch.Size([1, 44, 17, 17])
torch.Size([1, 45, 33, 33])
torch.Size([1, 45, 65, 65])
torch.Size([1, 45, 129, 129])
torch.Size([1, 3, 256, 256])


In [15]:
torch.manual_seed(seed)
random.seed(seed)

train_transforms = transforms.Compose(
    [transforms.RandomCrop(patch_size), transforms.ToTensor()]
)

test_transforms = transforms.Compose(
    [transforms.CenterCrop(patch_size), transforms.ToTensor()]
)

train_dataset = ImageFolder(image_dataset, transform=train_transforms)
test_dataset = SequenceFolder(sequence_dataset, split="test", transform=None)

In [16]:
test_dataset.__sizeof__()

24

In [17]:


device = "cuda" if cuda and torch.cuda.is_available() else "cpu"

train_dataloader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    num_workers=num_workers,
    shuffle=True,
    pin_memory=(device == "cuda"),
)

net = Network(N, M)
net = net.to(device)


optimizer = configure_optimizers(net, learning_rate)
lr_scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, "min")
criterion = Loss()

last_epoch = 0


best_loss = float("inf")

loss_epoch = [] # set up a list to store the loss value after each epoch
PSNR_epoch = [] # set up a list to store the loss value after each epoch

train_time = AverageMeter()

for epoch in range(last_epoch, epochs):
    print(f"Learning rate: {optimizer.param_groups[0]['lr']}")
    epoch_train_start = time.time()
    train_one_epoch(
        net,
        criterion,
        train_dataloader,
        optimizer,
        epoch,
        clip_max_norm,
    )
    epoch_train_end = time.time()
    train_time.update(epoch_train_end - epoch_train_start)
    loss,psnr= test_epoch(epoch, test_dataset, test_transforms, net, criterion)

    loss_epoch.append(loss) # save the loss value in the loss_epoch list
    PSNR_epoch.append(psnr)

    lr_scheduler.step(loss)

    is_best = loss < best_loss
    best_loss = min(loss, best_loss)

    if save:
        save_checkpoint(
            {
                "epoch": epoch,
                "state_dict": net.state_dict(),
                "loss": loss,
                "optimizer": optimizer.state_dict(),
                "lr_scheduler": lr_scheduler.state_dict(),
            },
            is_best,
        )
    
    


    
print('the overall training time (exclude testing) is {} min'.format(train_time.sum / 60))


Learning rate: 0.001
Train epoch 0: [0/1167 (0%)]	Loss: 9347.970 |	MSE loss: 36.659
Train epoch 0: [160/1167 (14%)]	Loss: 270.220 |	MSE loss: 1.060
Train epoch 0: [320/1167 (27%)]	Loss: 123.331 |	MSE loss: 0.484
Train epoch 0: [480/1167 (41%)]	Loss: 36.087 |	MSE loss: 0.142
Train epoch 0: [640/1167 (55%)]	Loss: 20.414 |	MSE loss: 0.080
Train epoch 0: [800/1167 (68%)]	Loss: 13.390 |	MSE loss: 0.053
Train epoch 0: [960/1167 (82%)]	Loss: 9.997 |	MSE loss: 0.039
Train epoch 0: [1120/1167 (96%)]	Loss: 9.233 |	MSE loss: 0.036
Test epoch 0: Average losses:	MSE loss: 0.039	Sequence-wise PSNR:  14.421

Learning rate: 0.001
Train epoch 1: [0/1167 (0%)]	Loss: 10.050 |	MSE loss: 0.039
Train epoch 1: [160/1167 (14%)]	Loss: 9.137 |	MSE loss: 0.036
Train epoch 1: [320/1167 (27%)]	Loss: 6.342 |	MSE loss: 0.025
Train epoch 1: [480/1167 (41%)]	Loss: 5.396 |	MSE loss: 0.021
Train epoch 1: [640/1167 (55%)]	Loss: 8.758 |	MSE loss: 0.034
Train epoch 1: [800/1167 (68%)]	Loss: 6.305 |	MSE loss: 0.025
Train ep